# AERIS — Random Forest Training and Evaluation

This is **stage 2** of the AERIS workflow. It consumes the processed-data contract
created by Notebook 1, tunes a Random Forest using engine-aware validation, checks the
selected model on held-out engines, and performs one final evaluation on NASA FD001.

No EDA, cleaning, or feature engineering is repeated here.

In [ ]:
from pathlib import Path
import json
import os
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit


working_dir = Path.cwd().resolve()
project_candidates = [working_dir, working_dir.parent, working_dir / "AERIS"]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "src" / "aeris").is_dir()),
    working_dir,
)

source_dir = PROJECT_ROOT / "src"
if source_dir.is_dir() and str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

try:
    from aeris import load_processed_data
except ModuleNotFoundError:
    print("AERIS package not found; using the standalone notebook loader.")

    def load_processed_data(data_dir):
        """Load Notebook 1 outputs when this notebook runs outside the repository."""
        data_dir = Path(data_dir)
        required_files = (
            "train_original.csv",
            "test_original.csv",
            "test_rul.csv",
            "dataset_metadata.json",
        )
        missing_files = [
            name for name in required_files if not (data_dir / name).is_file()
        ]
        if missing_files:
            raise FileNotFoundError(
                "Run Notebook 1 in this Colab runtime first, or place its outputs in "
                f"{data_dir}. Missing: {missing_files}"
            )

        train_df = pd.read_csv(data_dir / "train_original.csv")
        test_df = pd.read_csv(data_dir / "test_original.csv")
        test_rul_df = pd.read_csv(data_dir / "test_rul.csv")
        metadata = json.loads(
            (data_dir / "dataset_metadata.json").read_text(encoding="utf-8")
        )
        feature_columns = metadata["feature_columns"]

        return train_df, test_df, test_rul_df, feature_columns


base_data_dir = Path(
    os.getenv("AERIS_DATA_DIR", PROJECT_ROOT / "data")
).expanduser().resolve()
PROCESSED_DIR = base_data_dir / "processed"
MODEL_DIR = Path(
    os.getenv("AERIS_MODEL_DIR", PROJECT_ROOT / "models")
).expanduser().resolve()
REPORT_DIR = Path(
    os.getenv("AERIS_REPORT_DIR", PROJECT_ROOT / "reports")
).expanduser().resolve()

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Load the processed-data contract

`load_processed_data` is the single entry point for Notebook 1 outputs. It reads the
CSV files and metadata, validates their schema, and returns the exact feature order
that the model must use.

In [ ]:
train_df, test_df, test_rul_df, feature_columns = load_processed_data(PROCESSED_DIR)

print(f"Training data: {train_df.shape}")
print(f"Test data:     {test_df.shape}")
print(f"Features:      {len(feature_columns)}")
print(f"Data source:   {PROCESSED_DIR}")

## Step 2 — Reserve unseen engines for a holdout check

The split is performed by `engine_id`, never by individual rows. All cycles from one
engine therefore remain on the same side of the split. Hyperparameter selection uses
only the development engines; the holdout engines remain untouched until Step 4.

In [ ]:
X = train_df[feature_columns]
y = train_df["RUL"]
groups = train_df["engine_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
development_idx, holdout_idx = next(splitter.split(X, y, groups=groups))

X_development = X.iloc[development_idx]
y_development = y.iloc[development_idx]
development_groups = groups.iloc[development_idx]

X_holdout = X.iloc[holdout_idx]
y_holdout = y.iloc[holdout_idx]
holdout_groups = groups.iloc[holdout_idx]

development_engines = set(development_groups)
holdout_engines = set(holdout_groups)
assert development_engines.isdisjoint(holdout_engines)

print(f"Development engines: {len(development_engines)}")
print(f"Holdout engines:     {len(holdout_engines)}")
print(f"Development rows:    {len(X_development):,}")
print(f"Holdout rows:        {len(X_holdout):,}")

## Step 3 — Tune one reproducible search space

A single `GridSearchCV` replaces the two manual tuning loops. Three-fold
`GroupKFold` keeps engines separated during every comparison and selects the model
with the lowest mean validation MAE.

In [ ]:
parameter_grid = {
    "n_estimators": [50, 100],
    "max_depth": [10, 15, 20],
    "min_samples_leaf": [3, 5],
}

group_cv = GroupKFold(n_splits=3)
search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=parameter_grid,
    scoring="neg_mean_absolute_error",
    cv=group_cv,
    n_jobs=-1,
    return_train_score=False,
)
search.fit(X_development, y_development, groups=development_groups)

cv_results = pd.DataFrame(search.cv_results_)[
    [
        "param_n_estimators",
        "param_max_depth",
        "param_min_samples_leaf",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
    ]
].copy()
cv_results["mean_mae"] = -cv_results.pop("mean_test_score")
cv_results["mae_std"] = cv_results.pop("std_test_score")
cv_results = cv_results.sort_values("rank_test_score").reset_index(drop=True)

best_parameters = search.best_params_.copy()
display(cv_results.head(10).round(4))
print("Best parameters:", best_parameters)
print(f"Best grouped CV MAE: {-search.best_score_:.3f} cycles")

## Step 4 — Evaluate the selected model on held-out engines

This is the validation step missing from the earlier notebook. The selected Random
Forest is compared with a median baseline on engines that were not used by the search.
The NASA test set remains untouched.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "r2": r2_score(y_true, y_pred),
    }


selected_model = search.best_estimator_
holdout_prediction = selected_model.predict(X_holdout)

median_baseline = DummyRegressor(strategy="median")
median_baseline.fit(X_development, y_development)
baseline_prediction = median_baseline.predict(X_holdout)

holdout_metrics = pd.DataFrame(
    [
        {"model": "Median baseline", **regression_metrics(y_holdout, baseline_prediction)},
        {"model": "Selected Random Forest", **regression_metrics(y_holdout, holdout_prediction)},
    ]
)

holdout_predictions = pd.DataFrame({
    "engine_id": holdout_groups.to_numpy(),
    "actual_rul": y_holdout.to_numpy(),
    "predicted_rul": holdout_prediction,
})
holdout_predictions["absolute_error"] = np.abs(
    holdout_predictions["actual_rul"] - holdout_predictions["predicted_rul"]
)

display(holdout_metrics.round(4))

## Step 5 — Refit the selected configuration on all training engines

After the holdout check, the chosen configuration is trained once on the complete
FD001 training set. Hyperparameters are not changed after looking at NASA test results.

In [ ]:
final_model = RandomForestRegressor(
    **best_parameters,
    random_state=42,
    n_jobs=-1,
)
final_model.fit(X, y)

print("Final Random Forest trained.")
print(f"Training rows: {len(X):,}")
print(f"Features:      {len(feature_columns)}")

## Step 6 — Evaluate once on NASA FD001 test engines

NASA supplies one true RUL value for each test engine. The model therefore predicts
from the last observed cycle of every engine. In addition to MAE, RMSE, and R², the
asymmetric NASA score penalizes dangerous RUL overestimation more strongly.

In [ ]:
def nasa_score(y_true, y_pred):
    error = np.asarray(y_pred) - np.asarray(y_true)
    penalties = np.where(
        error < 0,
        np.exp(-error / 13.0) - 1.0,
        np.exp(error / 10.0) - 1.0,
    )
    return float(penalties.sum())


last_test_cycle = (
    test_df.sort_values(["engine_id", "cycle"])
    .groupby("engine_id", as_index=False)
    .tail(1)
    .sort_values("engine_id")
    .reset_index(drop=True)
)

y_test = test_rul_df["true_rul"].to_numpy()
if len(last_test_cycle) != len(y_test):
    raise ValueError(
        "The number of final test-engine rows does not match the number of true RUL values."
    )

test_prediction = final_model.predict(last_test_cycle[feature_columns])
test_metrics = {
    **regression_metrics(y_test, test_prediction),
    "nasa_score": nasa_score(y_test, test_prediction),
}

test_predictions = pd.DataFrame({
    "engine_id": last_test_cycle["engine_id"].to_numpy(),
    "actual_rul": y_test,
    "predicted_rul": test_prediction,
})
test_predictions["signed_error"] = (
    test_predictions["predicted_rul"] - test_predictions["actual_rul"]
)
test_predictions["absolute_error"] = test_predictions["signed_error"].abs()

display(pd.DataFrame([test_metrics]).round(4))
display(test_predictions.head(10).round(2))

## Step 7 — Inspect errors and feature importance

The diagnostics expose systematic bias, large engine-level mistakes, and the features
the forest uses most. Positive signed error means RUL was overestimated.

In [ ]:
feature_importance = (
    pd.DataFrame({
        "feature": feature_columns,
        "importance": final_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Overestimated engines:", int((test_predictions["signed_error"] > 0).sum()))
print("Underestimated engines:", int((test_predictions["signed_error"] < 0).sum()))
print(f"Mean signed error: {test_predictions['signed_error'].mean():.2f} cycles")
display(test_predictions.nlargest(10, "absolute_error").round(2))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

axes[0].scatter(test_predictions["actual_rul"], test_predictions["predicted_rul"], alpha=0.75)
limit = max(test_predictions["actual_rul"].max(), test_predictions["predicted_rul"].max())
axes[0].plot([0, limit], [0, limit], "--", color="black", linewidth=1)
axes[0].set(title="Actual vs predicted RUL", xlabel="Actual RUL", ylabel="Predicted RUL")

axes[1].hist(test_predictions["signed_error"], bins=15, edgecolor="black")
axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1].set(title="Test signed errors", xlabel="Predicted − actual", ylabel="Engines")

top_features = feature_importance.head(10).sort_values("importance")
axes[2].barh(top_features["feature"], top_features["importance"])
axes[2].set(title="Top feature importances", xlabel="Importance")

plt.tight_layout()
plt.show()

## Step 8 — Save the model and evaluation evidence

The model artifact stores the feature order and selected parameters with the estimator.
All validation and test tables are written to `reports/` for reproducibility.

In [ ]:
model_artifact = {
    "model": final_model,
    "features": feature_columns,
    "best_parameters": best_parameters,
    "dataset": "NASA C-MAPSS FD001",
}
joblib.dump(model_artifact, MODEL_DIR / "aeris_random_forest.joblib")

pd.DataFrame([test_metrics]).to_csv(REPORT_DIR / "nasa_test_metrics.csv", index=False)
test_predictions.to_csv(REPORT_DIR / "nasa_test_predictions.csv", index=False)
holdout_metrics.to_csv(REPORT_DIR / "holdout_metrics.csv", index=False)
holdout_predictions.to_csv(REPORT_DIR / "holdout_predictions.csv", index=False)
cv_results.to_csv(REPORT_DIR / "random_forest_cv_results.csv", index=False)
feature_importance.to_csv(REPORT_DIR / "feature_importance.csv", index=False)

print(f"Saved model to {MODEL_DIR}")
print(f"Saved evaluation reports to {REPORT_DIR}")

## Project status

This notebook now provides a complete, leakage-aware Random Forest baseline. It is
reproducible and suitable for comparison with future models, but it is not yet a
production prognostics system. Future work should include RUL capping experiments,
uncertainty estimates, additional FD subsets, and monitoring requirements.